In [ ]:
import pandas as pd
from rapidfuzz import fuzz

movies = pd.read_csv("../data/raw/music/10/title_basics.csv")
movies.set_index('tconst', inplace=True)
movie_dups = pd.read_csv("../data/raw/music/10/title_basics_dups.csv")
acts_in = pd.read_csv("../data/raw/music/10/title_principals.csv")

def calculate_record_similarity(row, records_df):
    """
    Computes a weighted similarity between two records.
    """
    r1 = records_df.loc[row['1']]
    r2 = records_df.loc[row['2']]
    
    # Calculate similarity for specific attributes
    # Using token_set_ratio is often better for ER (handles word reordering)
    title_sim = fuzz.token_set_ratio(r1['primaryTitle'], r2['primaryTitle']) / 100.0
    
    # Simple binary match for year
    year_sim = 1.0 if r1['startYear'] == r2['startYear'] else 0.0
    
    # Weighted average (Adjust weights based on your schema)
    return (title_sim * 0.8) + (year_sim * 0.2)

# 2. Compute similarity for all labeled duplicates
movie_dups['similarity_score'] = movie_dups.apply(
    calculate_record_similarity, 
    axis=1, 
    records_df=movies
)

# 3. Sort ascending to find "Hard Matches" 
# (Items labeled as duplicate but with low string similarity)
hard_matches = movie_dups.sort_values(by='similarity_score', ascending=True)

print("--- Hardest Matches (Low Similarity, but are Duplicates) ---")
for _, row in hard_matches.head(10).iterrows():
    r1 = movies.loc[row['1']]
    r2 = movies.loc[row['2']]
    print(f"Score: {row['similarity_score']:.2f}")
    print(f"  R1: {r1['primaryTitle']} ({r1['startYear']})")
    print(f"  R2: {r2['primaryTitle']} ({r2['startYear']})\n")

--- Hardest Matches (Low Similarity, but are Duplicates) ---
Score: 0.33
  R1: TheEven Money (2011)
  R2: Even Moneyb (2001)

Score: 0.53
  R1: HRi (2006)
  R2: HRg (2004)

Score: 0.60
  R1: Jessiel (2011)
  R2: TheJessie (2012)

Score: 0.62
  R1: TheKoichiro Unos Female Gymnastic Teacher (1987)
  R2: Koichiro Unos Female Gymnastic TeacherO (1979)

Score: 0.62
  R1: TheKoichiro Unos Female Gymnastic Teacher (1987)
  R2: Koichiro Unos Female Gymnastic Teacherv (1977)

Score: 0.62
  R1: TheRoxanne (2004)
  R2: RoxanneZ (2005)

Score: 0.64
  R1: HR (2007)
  R2: HRi (2006)

Score: 0.64
  R1: AmalP (2011)
  R2: Amalp (2016)

Score: 0.64
  R1: TheJessie (2012)
  R2: Jessie (2011)

Score: 0.64
  R1: HR (2006)
  R2: HRg (2004)



In [3]:
import random

def find_hard_non_matches(records_df, duplicates_df, sample_size=1000):
    """
    Finds non-duplicate pairs with high similarity (Hard Non-Matches).
    """
    # 1. Create a set of known duplicates for fast lookup (ground truth)
    # We store both (A, B) and (B, A) to ensure we don't pick them as non-matches
    known_dups = set()
    for _, row in movie_dups.iterrows():
        known_dups.add((row['1'], row['2']))
        known_dups.add((row['2'], row['1']))

    # 2. Blocking: Group by 'year' to find plausible non-match candidates
    # This prevents us from comparing "The Karate Kid" to "Finding Nemo"
    candidates = []
    grouped = movies.groupby('startYear')

    for year, group in grouped:
        if len(group) < 2:
            continue
            
        # Get IDs in this year block
        ids = group.index.tolist()
        
        # Sample pairs within the same year to find "Hard Non-Matches"
        # Increase 'k' if you want a more exhaustive search
        for _ in range(min(len(ids) * 2, 500)): 
            idx1, idx2 = random.sample(ids, 2)
            
            # Only proceed if they are NOT known duplicates
            if (idx1, idx2) not in known_dups:
                candidates.append((idx1, idx2))

    # 3. Calculate similarity for these candidates
    results = []
    for left_id, right_id in candidates[:sample_size]:
        r1 = movies.loc[left_id]
        r2 = movies.loc[right_id]
        
        # Scoring logic
        title_sim = fuzz.token_set_ratio(str(r1['primaryTitle']), str(r2['primaryTitle'])) / 100.0
        
        results.append({
            'left_id': left_id,
            'right_id': right_id,
            'similarity_score': title_sim,
            'left_title': r1['primaryTitle'],
            'right_title': r2['primaryTitle'],
            'year': r1['startYear']
        })

    # 4. Sort descending: High similarity + Not Duplicates = Hard Non-Matches
    hard_non_matches = pd.DataFrame(results).sort_values(by='similarity_score', ascending=False)
    
    return hard_non_matches

# Run the detection
hard_non_matches = find_hard_non_matches(movies, movie_dups)

print("--- Top 10 Hard Non-Matches (High Similarity, but NOT Duplicates) ---")
print(hard_non_matches[['left_title', 'right_title', 'similarity_score']].head(10))

--- Top 10 Hard Non-Matches (High Similarity, but NOT Duplicates) ---
                           left_title                         right_title  \
204              Bandits of the WestU  Inside the Walls of Folsom PrisonE   
195              Bandits of the WestU  Inside the Walls of Folsom PrisonE   
845             The House by the Lake    The Golden Claws of the Cat Girl   
962                         The Swapq                           The Devil   
961                         The Devil      The Education of Sonny Carsonz   
960    The Education of Sonny Carsonz                           The Devil   
392             Law of the Plainsmano                   Lola the Coalgirl   
577  The Golden Claws of the Cat Girl             The Island of Dr Moreau   
495                 The Golden Eagles                   Under the Lilacsy   
825             The House by the Lake                       The Ash Treei   

     similarity_score  
204          0.518519  
195          0.518519  
845       

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def evaluate_difficulty(movies_df, junction_df, duplicates_df, top_k=10):
    # 1. Prepare "Flattened" Record Strings for TF-IDF
    # Combine title and any other text features into one string
    movies_df['combined_text'] = movies_df['primaryTitle'].astype(str) + " " + movies_df.get('genres', '')
    
    tfidf = TfidfVectorizer(stop_words='english', analyzer='char_wb', ngram_range=(3,3))
    tfidf_matrix = tfidf.fit_transform(movies_df['combined_text'])
    
    # 2. Map Neighborhoods (Junction Table: movie_id -> [actor_ids])
    # Assuming junction_df has columns ['movie_id', 'neighbor_id']
    neighborhoods = junction_df.groupby('tconst')['nconst'].apply(set).to_dict()

    # 3. Create Ground Truth Set
    known_dups = set()
    for _, row in duplicates_df.iterrows():
        # Store as sorted tuple to handle (A,B) and (B,A) identically
        pair = tuple(sorted((row['1'], row['2'])))
        known_dups.add(pair)

    results = []
    movie_ids = movies_df.index.tolist()

    # 4. For a sample of movies, find the most similar others
    for i, m1_id in enumerate(movie_ids):
        # Get cosine similarity of this movie against ALL others
        sim_vector = cosine_similarity(tfidf_matrix[i], tfidf_matrix).flatten()
        
        # Get indices of top_k most similar (excluding itself)
        related_indices = sim_vector.argsort()[-(top_k+1):-1][::-1]
        
        for idx in related_indices:
            m2_id = movie_ids[idx]
            
            # --- THE CRITICAL CHECKS ---
            # 1. Identity Check: Skip if it's the same record
            if m1_id == m2_id:
                continue
            
            # 2. Symmetry Check: To avoid (A,B) and (B,A) duplicates in our results,
            # we only process if ID1 < ID2 (lexicographical order)
            if str(m1_id) >= str(m2_id):
                continue
            
            score = sim_vector[idx]
            
            # Neighborhood Jaccard Similarity
            n1 = neighborhoods.get(m1_id, set())
            n2 = neighborhoods.get(m2_id, set())
            
            intersection = len(n1.intersection(n2))
            union = len(n1.union(n2))
            jaccard = intersection / union if union > 0 else 0
            
            is_dup = (m1_id, m2_id) in known_dups or (m2_id, m1_id) in known_dups
            
            results.append({
                'id1': m1_id, 'id2': m2_id,
                'string_sim': score,
                'neighbor_sim': jaccard,
                'is_duplicate': is_dup
            })

    return pd.DataFrame(results)

# Run Analysis
diff_df = evaluate_difficulty(movies, acts_in, movie_dups)

# --- FINDING THE HARD CASES ---

# 1. Hard Matches (Duplicates with low similarity)
hard_matches = diff_df[diff_df['is_duplicate']].sort_values(['string_sim', 'neighbor_sim']).head(30)
display(hard_matches)

# 2. Hard Non-Matches (High similarity, but NOT duplicates)
hard_non_matches = diff_df[~diff_df['is_duplicate']].sort_values('string_sim', ascending=False).head(10)
display(hard_non_matches)

# 3. Neighborhood Conflict (Looks similar, but neighbors are totally different)
neighborhood_conflict = diff_df[(diff_df['string_sim'] > 0.4) & (diff_df['string_sim'] < 0.99)& (diff_df['neighbor_sim'] == 0)].sort_values('string_sim', ascending=False).head(10)
display(neighborhood_conflict)

/opt/anaconda3/envs/deepcem311/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:539: UserWarning: The parameter 'stop_words' will not be used since 'analyzer' != 'word'
  warnings.warn(


,id1,id2,string_sim,neighbor_sim,is_duplicate
8163,tt12538716,tt2390480,0.380600,0.625000,True
9958,tt12542969,tt12542970,0.383188,0.100000,True
13474,tt12532524,tt12532526,0.392339,0.000000,True
5924,tt12541274,tt12541275,0.397184,0.083333,True
10461,tt12531251,tt12531253,0.431204,0.000000,True
11531,tt12539052,tt12539054,0.443015,0.000000,True
795,tt12542968,tt12542969,0.443763,0.111111,True
2447,tt0920455,tt12532524,0.445788,0.363636,True
5074,tt12537876,tt12537877,0.448344,0.000000,True
3943,tt12542569,tt12542571,0.467475,0.142857,True


,id1,id2,string_sim,neighbor_sim,is_duplicate
5304,tt12536014,tt12539153,0.790566,0.0,False
11156,tt12539153,tt7941920,0.790566,0.0,False
11140,tt12538451,tt12539410,0.755027,0.0,False
4317,tt0299505,tt12539411,0.755027,0.0,False
4319,tt0299505,tt12539410,0.755027,0.0,False
4320,tt0299505,tt4601708,0.755027,0.0,False
11138,tt12538451,tt12539411,0.755027,0.0,False
12627,tt12538449,tt4601708,0.755027,0.0,False
12626,tt12538449,tt12539410,0.755027,0.0,False
12625,tt12538449,tt12539412,0.755027,0.0,False


,id1,id2,string_sim,neighbor_sim,is_duplicate
12894,tt12535275,tt3811966,0.980400,0.0,True
12895,tt12535275,tt12535276,0.980400,0.0,True
9977,tt12535537,tt12535538,0.970556,0.0,True
10965,tt0350750,tt12540574,0.969979,0.0,True
9476,tt12540574,tt12540575,0.969979,0.0,True
6341,tt12537300,tt12537301,0.968155,0.0,True
3888,tt12533048,tt12533049,0.965286,0.0,True
13390,tt12542408,tt12542410,0.963098,0.0,True
3156,tt12545610,tt12545611,0.962550,0.0,True
10113,tt12545609,tt12545610,0.962550,0.0,True


In [1]:
my_list = [3,5,9]
print(sum(my_list))


17
